# Flan - Verificacion Funcional (explicacion interactiva)

Capsula 3. Ya tenemos el RTL del semaforo... pero **hace lo correcto?**
Aqui lo comprobamos con un **testbench**, vemos el **checker** en accion y
-lo mas divertido- **inyectamos un bug** para ver como la verificacion lo caza.

> Corre las celdas en orden (Shift+Enter).

## 1 - Preparar el entorno

In [ ]:
import os, subprocess
!apt-get -qq update && apt-get -qq install -y iverilog > /dev/null
import matplotlib.pyplot as plt
REPO_URL = 'https://github.com/Misterio12947/galletas-y-postres-chips.git'
REPO = 'galletas-y-postres-chips'
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL
else:
    !cd $REPO && git pull --ff-only
RTL = f'{REPO}/common/rtl/semaphore/rtl/semaphore.v'          # el RTL CORRECTO
TB  = f'{REPO}/common/rtl/semaphore/tb/tb_semaphore.v'        # el testbench
assert os.path.exists(RTL) and os.path.exists(TB)
print('Listo.')

## 2 - Anatomia de un testbench

Verificar no es 'ver ondas': es **comparar contra lo esperado**. Un testbench
tiene tres piezas:

- **Estimulo** - genera las entradas (aqui: el reloj y el reset)
- **Checker** - compara la salida contra lo que deberia pasar (aqui: *nunca dos luces a la vez*)
- **Cobertura** - mide cuanto del diseno probamos (vimos los 3 estados?)

Este es el testbench real del repo. Fijate en el `always` del checker:

In [ ]:
print(open(TB).read())

## 3 - Verificar el RTL correcto

Lo simulamos: debe imprimir `PASS` y, en la onda, **solo una luz encendida**
en cada momento.

In [ ]:
def _parse_vcd(path, names):
    sym2name, ch, t = {}, {n: [] for n in names}, 0
    for ln in open(path):
        ln = ln.strip()
        if ln.startswith('$var'):
            p = ln.split(); sym2name[p[3]] = p[4]
        elif ln.startswith('#'):
            t = int(ln[1:])
        elif ln and ln[0] in '01xz' and len(ln) >= 2:
            nm = sym2name.get(ln[1:])
            if nm in names: ch[nm].append((t, 1 if ln[0]=='1' else 0))
    return ch

def _plot(ch, titulo=''):
    tmax = max((tv[-1][0] for tv in ch.values() if tv), default=1) + 10
    fig, axes = plt.subplots(3, 1, figsize=(10, 2.4), sharex=True)
    for ax, nm, c in zip(axes, ['green','yellow','red'], ['tab:green','gold','tab:red']):
        xs, ys = [0], [0]
        for tt, vv in ch[nm]: xs.append(tt); ys.append(vv)
        xs.append(tmax); ys.append(ys[-1])
        ax.step(xs, ys, where='post', color=c, lw=2)
        ax.set_ylim(-0.2,1.2); ax.set_yticks([0,1])
        ax.set_ylabel(nm, rotation=0, ha='right', va='center')
    axes[0].set_title(titulo); axes[-1].set_xlabel('tiempo (ns)')
    plt.tight_layout(); plt.show()

def correr(rtl_file, verde=3, amarillo=2, rojo=3, total=500):
    tb = f'''`timescale 1ns/1ps
module tb;
  reg clk=0, rst_n=0; wire red, yellow, green;
  semaforo #(.T_GREEN({verde}), .T_YELLOW({amarillo}), .T_RED({rojo}), .CW(8)) dut(
    .clk(clk), .rst_n(rst_n), .red(red), .yellow(yellow), .green(green));
  always #5 clk = ~clk;
  integer errors = 0;
  always @(posedge clk) if (rst_n) if ((red+yellow+green)!==1) errors=errors+1;
  initial begin
    $dumpfile("d.vcd"); $dumpvars(0, tb);
    #12 rst_n=1; #{total};
    if (errors==0) $display(">>> PASS: una sola luz a la vez");
    else           $display(">>> FAIL: %0d violaciones (el checker las cazo)", errors);
    $finish;
  end
endmodule
'''
    open('tb_gen.v','w').write(tb)
    r = subprocess.run(['bash','-lc', f'iverilog -g2012 -o s {rtl_file} tb_gen.v && vvp s'],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return _parse_vcd('d.vcd', ['green','yellow','red'])

ch = correr(RTL)
_plot(ch, 'RTL correcto -> PASS')

## 4 - El checker en accion: inyecta un bug

Preparamos una version del semaforo **con un bug a proposito**: en AMARILLO
tambien enciende el verde (dos luces a la vez). Selecciona abajo y mira como
el **checker lo caza** (`FAIL`) y como en la onda se ven **dos luces juntas**.

In [ ]:
bug = '''`timescale 1ns/1ps
// BUG a proposito: en AMARILLO tambien enciende el verde
module semaforo #(parameter integer T_GREEN=5, T_YELLOW=2, T_RED=5, CW=8)(
  input wire clk, input wire rst_n, output reg red, output reg yellow, output reg green);
  localparam [1:0] S_GREEN=0, S_YELLOW=1, S_RED=2;
  reg [1:0] state; reg [CW-1:0] cnt, limit;
  always @(*) case(state) S_GREEN:limit=T_GREEN; S_YELLOW:limit=T_YELLOW; default:limit=T_RED; endcase
  always @(posedge clk or negedge rst_n)
    if(!rst_n) begin state<=S_GREEN; cnt<=0; end
    else if(cnt==limit-1) begin cnt<=0;
      state<=(state==S_GREEN)?S_YELLOW:(state==S_YELLOW)?S_RED:S_GREEN; end
    else cnt<=cnt+1;
  always @(*) begin red=0; yellow=0; green=0;
    case(state)
      S_GREEN:  green=1;
      S_YELLOW: begin yellow=1; green=1; end   // <-- BUG: dos luces
      S_RED:    red=1;
    endcase
  end
endmodule
'''
open('semaphore_bug.v','w').write(bug)

from ipywidgets import interact
def demo(version='RTL correcto'):
    f = RTL if version == 'RTL correcto' else 'semaphore_bug.v'
    ch = correr(f)
    _plot(ch, version)
interact(demo, version=['RTL correcto', 'RTL con bug']);

## 5 - Cobertura: probamos todo?

El `PASS` dice que lo que probamos salio bien... pero **probamos suficiente?**
Veamos cuantos de los 3 estados alcanzamos a visitar:

In [ ]:
ch = correr(RTL)
colores = {'green':'VERDE', 'yellow':'AMARILLO', 'red':'ROJO'}
visitados = [colores[c] for c in ['green','yellow','red'] if any(v==1 for _,v in ch[c])]
print(f'Estados visitados: {len(visitados)}/3  ->  {visitados}')
print('Cobertura de estados:', '100%' if len(visitados)==3 else f'{len(visitados)*100//3}%')
print('\n(En UVM esto se mide automatico con covergroups; aqui lo hacemos a mano para entenderlo.)')

## 6 - Para pensar / lo que sigue

- El `PASS` prueba los casos que **simulamos**. Y los que no se nos ocurrieron?
- Por eso existe la **verificacion formal**: demostrar la propiedad para **TODOS**
  los casos, sin depender de que el testbench los visite.

🍮 Flan &nbsp;->&nbsp; 🟦 Gelatina (Verificacion Formal)